In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [38]:
df = pd.read_csv("data.csv")

In [46]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.8,4.0,1.2,0.2,setosa
1,5.7,4.4,1.5,0.4,setosa
2,5.4,3.9,1.3,0.4,setosa
3,5.1,3.5,1.4,0.3,setosa
4,5.7,3.8,1.7,0.3,setosa


In [42]:
X = df.drop(columns=["species"])
y = df["species"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
print(X_train.shape, X_test.shape)

(80, 4) (21, 4)


In [7]:
import itertools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [8]:
n_estimators_grid = [50, 100, 200]
max_depth_grid = [2, 4, None]

In [9]:
results = []

for n_estimators, max_depth in itertools.product(n_estimators_grid, max_depth_grid):
    params = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": 42,
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, average="macro"),
        "recall": recall_score(y_test, preds, average="macro"),
        "f1": f1_score(y_test, preds, average="macro"),
    }

    results.append({"params": params, "metrics": metrics, "model": model})
    print(params, "->", metrics)

{'n_estimators': 50, 'max_depth': 2, 'random_state': 42} -> {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
{'n_estimators': 50, 'max_depth': 4, 'random_state': 42} -> {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
{'n_estimators': 50, 'max_depth': None, 'random_state': 42} -> {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
{'n_estimators': 100, 'max_depth': 2, 'random_state': 42} -> {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
{'n_estimators': 100, 'max_depth': 4, 'random_state': 42} -> {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
{'n_estimators': 100, 'max_depth': None, 'random_state': 42} -> {'accuracy': 0.952380

In [10]:
summary = pd.DataFrame([
    {**r["params"], **r["metrics"]} for r in results
])
summary.sort_values("f1", ascending=False)

,n_estimators,max_depth,random_state,accuracy,precision,recall,f1
0,50,2.0,42,0.952381,0.958333,0.944444,0.947475
1,50,4.0,42,0.952381,0.958333,0.944444,0.947475
2,50,NaN,42,0.952381,0.958333,0.944444,0.947475
3,100,2.0,42,0.952381,0.958333,0.944444,0.947475
4,100,4.0,42,0.952381,0.958333,0.944444,0.947475
5,100,NaN,42,0.952381,0.958333,0.944444,0.947475
6,200,2.0,42,0.952381,0.958333,0.944444,0.947475
7,200,4.0,42,0.952381,0.958333,0.944444,0.947475
8,200,NaN,42,0.952381,0.958333,0.944444,0.947475


In [11]:
import sys
!{sys.executable} -m pip install mlflow

In [12]:
import mlflow

In [13]:
mlflow.set_tracking_uri("http://localhost:8100")
client = mlflow.MlflowClient()
client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1784651305277, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784651305277, lifecycle_stage='active', name='iris-hpo', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1784648960453, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1784648960453, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [14]:
mlflow.set_experiment("iris-hpo")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1784651305277, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784651305277, lifecycle_stage='active', name='iris-hpo', tags={}, trace_location=None, workspace='default'>

In [15]:
from mlflow.models import infer_signature

best_f1, best_run_id = -1, None

for n_estimators, max_depth in itertools.product(n_estimators_grid, max_depth_grid):
    with mlflow.start_run(run_name=f"rf_n{n_estimators}_d{max_depth}") as run:
        params = {"n_estimators": n_estimators, "max_depth": max_depth, "random_state": 42}
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds, average="macro"),
            "recall": recall_score(y_test, preds, average="macro"),
            "f1": f1_score(y_test, preds, average="macro"),
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        signature = infer_signature(X_train, model.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=model,
            name="iris_model",
            signature=signature,
            input_example=X_train.iloc[:5],
        )

        print(run.info.run_id, params, metrics)

        if metrics["f1"] > best_f1:
            best_f1, best_run_id = metrics["f1"], run.info.run_id

print("Best run:", best_run_id, "F1:", best_f1)

e8bb829efd394ff985a45ed1ed8b580e {'n_estimators': 50, 'max_depth': 2, 'random_state': 42} {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
🏃 View run rf_n50_d2 at: http://localhost:8100/#/experiments/1/runs/e8bb829efd394ff985a45ed1ed8b580e
🧪 View experiment at: http://localhost:8100/#/experiments/1
f1735145d8984981bb0b1d5b27039ceb {'n_estimators': 50, 'max_depth': 4, 'random_state': 42} {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
🏃 View run rf_n50_d4 at: http://localhost:8100/#/experiments/1/runs/f1735145d8984981bb0b1d5b27039ceb
🧪 View experiment at: http://localhost:8100/#/experiments/1
282667f3dad04f678fb277ff1aa2d60e {'n_estimators': 50, 'max_depth': None, 'random_state': 42} {'accuracy': 0.9523809523809523, 'precision': 0.9583333333333334, 'recall': 0.9444444444444445, 'f1': 0.9474747474747476}
🏃 View run rf_n50_dNone at: http://loca

In [16]:
model_uri = f"runs:/{best_run_id}/iris_model"
registered = mlflow.register_model(model_uri=model_uri, name="iris-classifier")
client.set_registered_model_alias("iris-classifier", "champion", registered.version)
print("Registered version", registered.version, "as champion")

Successfully registered model 'iris-classifier'.
2026/07/21 16:31:23 WARNING mlflow.tracking._model_registry.fluent: Run with id e8bb829efd394ff985a45ed1ed8b580e has no artifacts at artifact path 'iris_model', registering model based on models:/m-016c3fd20bb04933bfcef2eec41e891f instead
2026/07/21 16:31:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


Registered version 1 as champion


In [22]:
!dvc remove models/*.dvc

/bin/bash: line 1: dvc: command not found


In [24]:
!which 
!{sys.executable} -m dvc --version

/opt/micromamba/envs/jupyterlab/bin/python3: No module named dvc


In [25]:
import sys
print(sys.executable)

/opt/micromamba/envs/jupyterlab/bin/python3


In [26]:
!micromamba env list

  Name        Active  Path                           
───────────────────────────────────────────────────────
  base        *       /opt/micromamba                
  jupyterlab          /opt/micromamba/envs/jupyterlab


In [28]:
%cd ~
!dvc remove model/iris_model.pkl.dvc

/home/jupyter


In [29]:
!echo "model/" >> .gitignore
!cat .gitignore

.cache/
.config/
.docker/
.gemini/
.gsutil/
.ipynb_checkpoints/
.ipython/
.jupyter/
.local/
.npm/
.bash_history
.lastactive
iris_train.csv
iris_eval.csv
model.pkl
model_loaded.pkl
training_log.json
.env/
__pycache__/
*.pyc
*.joblib
feature_repo/__pycache__/
feature_repo/bq/data/
feature_repo/data/*.db
model/


In [30]:
!git status

On branch week_5
Your branch is up to date with 'origin/week_5'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   22DP1000105_Assignment_5_MAY_2026_MLOps.ipynb
	deleted:    model/iris_model.pkl.dvc

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitconfig
	.lesshst

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
!git add .gitignore model/iris_model.pkl.dvc 22DP1000105_Assignment_5_MAY_2026_MLOps.ipynb

The following paths are ignored by one of your .gitignore files:
model
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"


In [32]:
!git commit -m "Task 4: remove model tracking from DVC, use MLflow registry instead"

[week_5 8be12f8] Task 4: remove model tracking from DVC, use MLflow registry instead
 3 files changed, 280 insertions(+), 21 deletions(-)
 delete mode 100644 model/iris_model.pkl.dvc


# Task 5

In [33]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri("http://localhost:8100")

model_uri = "models:/iris-classifier@champion"
champion_model = mlflow.pyfunc.load_model(model_uri)

# Load your eval data (adjust path/columns as needed)
eval_df = pd.read_csv("data.csv")
X_eval = eval_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y_true = eval_df["species"]

preds = champion_model.predict(X_eval)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_true, preds))
print("Precision (macro):", precision_score(y_true, preds, average="macro"))
print("Recall (macro):", recall_score(y_true, preds, average="macro"))
print("F1 (macro):", f1_score(y_true, preds, average="macro"))

Accuracy: 0.9603960396039604
Precision (macro): 0.961068211068211
Recall (macro): 0.9571428571428572
F1 (macro): 0.9584929757343549


In [35]:
!pip show pytest